# 第3章 前馈神经网络 · 教学练习



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter3/Chapter3.md]]（神经元、激活、逻辑门、前向与反向传播、万能近似定理）
- [[课程/深度学习/PPT/Chapter3(1)/Chapter3(1).md]]（补充：向量输出 Jacobian 与标量损失梯度）
- [[课程/深度学习/课堂笔记/2026-09-14-第5节-神经网络反向传播算法推导与复杂度分析讲座.md]]（第三至九节）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第二、三节：逼近存在性与训练的区别）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 2、3、7；保留任务，明确数组容器校注）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C3-01、03—11、A-04、05、15）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（后续自动微分示范，并非独立的第3章演示）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分、无代码补全及API范围）


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## Q301 激活函数：数值、导数与零点约定

**新编，依据 Chapter3“常见激活函数”、汇编 C3-01。**

实现 `activation(z, kind)`，返回同形状的 `(a, da)`，支持 `ReLU`、`tanh`（大小写不敏感）。
ReLU 导数在负数为0、正数为1，**本练习约定0处取0**；tanh 导数为 $1-a^2$。
在 `z=[-3,-1,0,1,3]` 上打印数值，解释为何 ReLU 的零点导数不是普通可导意义的导数。
用中心差分检查 `z=[-2,-.5,.5,2]`，步长 $10^{-6}$，不要在0处比较差分与约定导数。

课件以 $H(0)=1$ 表示弱导数的一个代表；单点取值不改变弱导数的等价类。
本练习的实现取0，不能把两种单点约定混写成同一个数值结论。


<details><summary>提示（先尝试再展开）</summary>

先计算激活值，tanh 导数直接复用激活值；ReLU 的掩码必须是 `z>0`。


</details>


In [ ]:
def activation(z, kind):
    # 返回激活与局部导数；在函数内实现，不要直接运行未完成函数。
    raise NotImplementedError("实现 ReLU/tanh")


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    z = np.array([-1., 0., 2.])
    a, da = activation(z, 'ReLU')
    assert np.allclose(a, [0., 0., 2.]) and np.allclose(da, [0., 0., 1.])
    a, da = activation(z, 'tanh')
    assert np.allclose(a, np.tanh(z)) and np.allclose(da, 1-np.tanh(z)**2)
    for kind in ('relu', 'tanh'):
        z = np.array([-.7, .4, 1.2]); h = 1e-6
        fd = (activation(z+h, kind)[0]-activation(z-h, kind)[0])/(2*h)
        assert np.max(np.abs(fd-activation(z, kind)[1])) < 1e-7
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q302 由神经元组合得到 AND、OR、XOR

**新编代码化，依据 Chapter3“用神经元学习二元逻辑函数”、汇编 C3-03。**

输入行依次为 `(0,0),(1,0),(0,1),(1,1)`。定义 $H(t)=1$ 当 $t\geq0$，否则0。
编写 `logic_gates(X)` 返回 `(and_y, or_y, xor_y)`。
AND、OR 各用一个神经元；XOR 用两层组合，不得直接使用位运算/逻辑异或。
打印两个隐藏神经元和最后输出。写出单个线性阈值不能表示 XOR 的四个不等式，并相加导出矛盾。


<details><summary>提示（先尝试再展开）</summary>

先用两隐藏神经元得到 OR、AND，输出对二者作阈值组合。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    X = np.array([[0,0],[1,0],[0,1],[1,1]])
    a,o,q = logic_gates(X)
    assert np.allclose(a, [0,0,0,1])
    assert np.allclose(o, [0,1,1,1])
    assert np.allclose(q, [0,1,1,0])
    p = np.array([3,0,2,1]); ap,op,qp = logic_gates(X[p])
    assert np.allclose(qp, q[p])
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q303 批量前向与参数量，不把输入维当批量维

**新编，依据 Chapter3“全连接神经网络”、Homework02 Exercise 2、汇编 A-04。**

实现 `forward_batch(X, W, b)`，行是样本，隐藏层全部为 ReLU，输出层恒等。
`W[l]` 采用 `(out,in)`，`b[l]` 为 `(out,)`；返回每层激活列表（含输入）和参数总数。
测试网络 `3→2→4→1`、5个输入；再用全部权重为1、偏置为0的 `2→3→1` 网络做能手算的例子。
打印矩阵乘法形状，说明为什么批量实现是 `X@W.T+b` 而不是 `W@X`。


<details><summary>提示（先尝试再展开）</summary>

一层参数数目是 `out*in+out`；输出恒等，负的输出偏置也必须保留。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    s,n = forward_batch(np.array([[1.,2.],[-1.,-2.],[0.,0.]]),
                        [np.ones((3,2)),np.ones((1,3))],[np.zeros(3),np.zeros(1)])
    assert n == 13 and np.allclose(s[-1][:,0], [9.,0.,0.])
    assert [a.shape for a in s] == [(3,2),(3,3),(3,1)]
    s,n = forward_batch(np.array([[0.,0.]]), [np.zeros((1,2))],[np.array([2.])])
    assert n == 3 and np.allclose(s[-1], [[2.]])
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q304 参数不同而函数相同：缩放与置换

**新编，依据9月14日课堂笔记第二节、汇编 C3-04。**

给定 $f(x)=v^T\operatorname{ReLU}(Wx+b)+c$。
编写 `equivalent_parameters(W,b,v,c,scale,perm)`，先将所有隐藏神经元的输入权重和偏置乘正数 `scale`，
相应输出权重除该数，再同步置换隐藏神经元，返回新参数。
用 `W=[[1,-2],[-.5,1]]`、`b=[.3,-.2]`、`v=[2,-3]`、`c=.4`、`scale=3`、`perm=[1,0]`，
在100个随机输入检查输出不变；推导对所有输入均相同的理由。额外说明负缩放为何不适用。

### 自检接口／需保存的结果

`equivalent_parameters(...)` 按顺序返回四元组 `(Wnew,bnew,vnew,cnew)`，各项类型、形状与输入的 `W,b,v,c` 对应；`Wnew` 为二维NumPy数组，`bnew,vnew` 为一维数组，`cnew` 为标量。自检自行创建输入，不要求额外的全局结果名。


<details><summary>提示（先尝试再展开）</summary>

ReLU 的正齐次性是 `ReLU(s*z)=s*ReLU(z)`，只对 `s>0` 使用。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    Xc = np.array([[0.,0.],[1.,-1.],[-2.,3.]])
    Wc=np.array([[1.,-2.],[-.5,1.]]); bc=np.array([.3,-.2]); vc=np.array([2.,-3.])
    wn,bn,vn,cn=equivalent_parameters(Wc,bc,vc,.4,2.,[1,0])
    assert not np.allclose(wn,Wc)
    assert np.allclose(np.maximum(Xc@Wc.T+bc,0)@vc+.4, np.maximum(Xc@wn.T+bn,0)@vn+cn)
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q305 Homework02 Exercise 3：手动前后向

**原作业数值复现并增加公开自检，不是新编真题。**

令 `A=[[2,1,-1],[-2,1,-1]]`、`B=[[1,-2],[2,-1],[0,1]]`、`b=[1,2,3]`、`v=[1,1,2]`。
网络为 $f(x)=v^T(B\operatorname{ReLU}(Ax)+b)^2$，平方逐元素。
在 `x=[1,2,3]` 手动计算函数值、输入导数及全部**实际出现参数** `A,B,b,v` 的导数。
编写 `homework3(x,A,B,b,v)` 返回字典，键为 `f,dx,dA,dB,db,dv`；不得用自动微分或有限差分代替反传。
打印预激活、隐藏输出、反向量。原公式不含第一层偏置/输出偏置，不能悄悄增加待求参数。

### 自检接口／需保存的结果

`homework3(...)` 返回的字典中，`f` 为标量；`dx,dA,dB,db,dv` 均为NumPy数组，形状依次为 `(3,),(2,3),(3,2),(3,),(3,)`。自检自行创建题面参数，无须保存额外的全局结果变量。


<details><summary>提示（先尝试再展开）</summary>

先得到 `z1=(1,-3)`；反传从 `p2=2*v*z2` 开始，再乘 B 的转置及 ReLU 掩码。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    A=np.array([[2.,1.,-1.],[-2.,1.,-1.]]); B=np.array([[1.,-2.],[2.,-1.],[0.,1.]])
    b=np.array([1.,2.,3.]); v=np.array([1.,1.,2.]); x=np.array([1.,2.,3.])
    r=homework3(x,A,B,b,v)
    assert abs(r['f']-38)<1e-12 and np.allclose(r['dx'],[40,20,-20])
    assert np.allclose(r['dA'],[[20,40,60],[0,0,0]])
    assert np.allclose(r['dB'],[[4,0],[8,0],[12,0]])
    assert np.allclose(r['db'],[4,8,12]) and np.allclose(r['dv'],[4,16,9])
    h=1e-6; Ap=A.copy(); Am=A.copy(); Ap[0,1]+=h; Am[0,1]-=h
    fd=(homework3(x,Ap,B,b,v)['f']-homework3(x,Am,B,b,v)['f'])/(2*h)
    assert abs(fd-r['dA'][0,1])<1e-6
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q306 自动微分校准：标量损失与梯度累积

**新编，依据9月14日“从手推反向传播到自动微分”、9月21日第四节与第4章演示。**

取 $z=wx+b,h=\operatorname{ReLU}(z),\hat y=ah+c,\ell=(\hat y-t)^2$。
设 `x=2,w=1,b=0,a=3,c=0,t=1`，所有待求参数用 float64 叶张量表示。
保存一次反传的梯度 `first306`（顺序 w,b,a,c），重新前向不清梯度再反传，保存 `twice306`；
清除 `.grad` 后再重新前向反传，保存 `reset306`。解释 `backward()` 与 `step()` 不同。
不要在旧图上强行反传两次，也不要把 `.grad` 的引用当作独立快照。


### 自检接口／需保存的结果

除已要求的三个梯度快照外，保存 `parameters306=[w,b,a,c]`：按此顺序存放四个CPU、float64标量叶张量的list，保留实际参数对象，供自检确认反传没有更新参数。`first306,twice306,reset306` 均保存为形状 `(4,)` 的NumPy数值数组，而不是 `.grad` 的共享引用。


<details><summary>提示（先尝试再展开）</summary>

每次调用一个重新构造损失的函数。`.item()` 得到数值快照；`p.grad=None` 是清除累积的一种方式。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert np.allclose(first306,[60,30,20,10])
    assert np.allclose(twice306,2*first306) and np.allclose(reset306,first306)
    assert all(abs(p.item()-v)<1e-12 for p,v in zip(parameters306,[1,0,3,0]))
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q307 任意层宽的 NumPy propagation：真正的反向传播

**Homework02 Exercise 7(1) 的实现练习，长作业扩展，非30分钟测试。**

严格实现接口 `propagation(x,W_list,b_list,activation_type) → fx,DfDx,DfDW,DfDb`。
输入是1D向量，权重采用 `(out,in)`，各层偏置1D；所有隐藏层共用 ReLU 或 tanh，输出层恒等且宽度为1。
返回 `fx` 为数值，`DfDx` 为1D数组，**`DfDW`、`DfDb` 用list，元素与对应参数同形状**。
原作业把它们称为3D/2D-array，但层宽不等时无法组成普通矩形高维数组；这里作明确容器校注，不填充伪造梯度。

先缓存前向值，再以输出种子1逆序回传；不得用有限差分或 PyTorch 替代。
应支持无隐藏层的线性网络，拒绝不兼容层形状和未知激活。ReLU 在0处约定导数0。
用 `2→2→1` 网络验证数值，并说明梯度外积的两个因子各来自哪里。

### 自检接口／需保存的结果

保留题面四项返回顺序；`DfDx.shape==(d,)`，`DfDW[l].shape==W_list[l].shape`，`DfDb[l].shape==b_list[l].shape`，两个逐层梯度容器必须是list。无隐藏层时仍返回各含一个元素的梯度list；层形状不兼容时抛出 `ValueError`，未知激活也应拒绝。自检自行创建小网络，不要求额外的全局样例变量。


<details><summary>提示（先尝试再展开）</summary>

反向递推：`delta_prev=(W.T@delta)*activation_derivative`；当前权重梯度为 `np.outer(delta,a_prev)`，偏置梯度为 `delta.copy()`。最后一层不乘隐藏激活导数。


</details>


In [ ]:
def propagation(x, W_list, b_list, activation_type):
    # 完成前向缓存与逆序反向循环；层梯度必须返回 list。
    raise NotImplementedError("任意层宽的前后向传播")


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    Wc=[np.array([[1.,2.],[-1.,1.]]),np.array([[2.,-3.]])]
    bc=[np.array([.5,-.5]),np.array([1.])]
    f,dx,dw,db=propagation(np.array([1.,2.]),Wc,bc,'ReLU')
    assert abs(f-10.5)<1e-12 and np.allclose(dx,[5,1])
    assert isinstance(dw,list) and isinstance(db,list)
    assert np.allclose(dw[0],[[2,4],[-3,-6]]) and np.allclose(dw[1],[[5.5,.5]])
    assert np.allclose(db[0],[2,-3]) and np.allclose(db[1],[1])
    f,dx,dw,db=propagation(np.array([2.,-1.]),[np.array([[3.,4.]])],[np.array([.5])],'tanh')
    assert abs(f-2.5)<1e-12 and np.allclose(dx,[3,4])
    f,dx,dw,db=propagation(np.array([0.]),[np.ones((1,1)),np.ones((1,1))],[np.zeros(1),np.zeros(1)],'ReLU')
    assert dx[0]==0 and dw[0][0,0]==0 and db[0][0]==0
    try:
        propagation(np.array([1.,2.]),[np.ones((1,3))],[np.zeros(1)],'relu')
    except ValueError:
        pass
    else:
        raise AssertionError('shape mismatch must be rejected')
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q308 Homework02 Exercise 7(2)—(4)：五输入与四坐标差分

**原作业要求完整落地；长作业扩展，建议45—75分钟，可分次完成。**

复用或重新实现 `propagation`（答案在本题单元内也独立定义）。取 $d=3,L=5$、隐藏层宽依次为 **2,4,6,8,10**，
输出维数1。固定种子308，权重均匀采样 `[-.15,.15]`；隐藏层偏置按索引0,1,…交替为 `.6,-.6`，输出偏置 `.2`。
输入为同一生成器继续采样得到的5个三维向量，范围 `[-.6,.6]`。

1. 对 ReLU、tanh 两种激活分别输出5个输入的函数值、全部输入梯度、各层全部权重与偏置梯度（保留 list）。
2. 用中心差分 $h=10^{-5}$ 比较指定的**一始下标**：$x_1,w^1_{11},w^2_{22},b^3_3$。
   对应 Python 下标 `dx[0],dw[0][0,0],dw[1][1,1],db[2][2]`；保存40条 `records308`，
   每行 `(激活,输入编号,坐标名,解析值,差分值,绝对误差,缩放误差)`。
3. 对 ReLU，核对每个扰动前后所有隐藏层的激活正负模式一致，避免跨越折点。
4. 讨论若对全部 $P$ 个参数及 $d$ 个输入坐标差分，需要多少次前向；与一次前向加反向比较。

允许教学扩展 API：`np.array_equal` 用于逐层比较符号模式；不得用 autograd 代替手动反传。


### 自检接口／需保存的结果

本题作答单元中须可调用 `propagation`，并保存以下结果，供自检读取：

- `sizes308`：Python list，内容为 `[3,2,4,6,8,10,1]`。
- `W308`：含6个NumPy矩阵的list，形状依次为 `(2,3),(4,2),(6,4),(8,6),(10,8),(1,10)`；`b308`：含6个一维数组的list，形状依次为 `(2,),(4,),(6,),(8,),(10,),(1,)`。使用题面给定的采样范围与偏置设置。
- `X308`：形状 `(5,3)` 的NumPy数组；在同一seed=308生成器中，先按层生成全部权重，再采样这5个输入。
- `records308`：含40个七元组的list，列含义沿用题面；激活名称用 `ReLU` 或 `tanh`，输入编号用1—5，坐标名称分别用 `x1,w11_1,w22_2,b3_3`。解析值、差分值及两种误差均保存为普通数值；绝对误差放第6列（Python索引5），缩放误差取绝对误差除以 `max(1,abs(解析值),abs(差分值))`。

其余缓存、符号模式与差分辅助函数可自定名称，不要求照抄参考答案的内部命名。


<details><summary>提示（先尝试再展开）</summary>

先把四个数学坐标准确转换为 Python 坐标；每次扰动都拷贝参数，防止累积污染。ReLU 关闭通道的导数为0是正确结果，tanh组能同时提供非零检验。


</details>


In [ ]:
# 建议先完成 Q307，再把自己的 propagation 粘到本单元（不要依赖答案文件）。
# TODO：构造6层参数、5个输入；逐激活输出完整梯度和40条差分比较 records308。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert sizes308 == [3,2,4,6,8,10,1] and X308.shape == (5,3)
    assert [w.shape for w in W308] == [(2,3),(4,2),(6,4),(8,6),(10,8),(1,10)]
    assert len(records308)==40 and {r[2] for r in records308}=={'x1','w11_1','w22_2','b3_3'}
    assert max(r[5] for r in records308)<1e-7
    assert any(abs(r[3])>1e-8 for r in records308 if r[0]=='tanh')
    for kind in ['ReLU','tanh']:
        for x in X308:
            f,dx,dw,db=propagation(x,W308,b308,kind)
            assert len(dw)==len(db)==6 and dx.shape==(3,)
            assert all(g.shape==w.shape for g,w in zip(dw,W308))
            assert all(g.shape==b.shape for g,b in zip(db,b308))
            # independent directional check mixes all parameters and input coordinates
            h=1e-5
            direction_x=np.array([.2,-.3,.1])
            direction_w=[np.ones_like(w)*.01 for w in W308]
            direction_b=[np.ones_like(b)*.02 for b in b308]
            predicted=dx@direction_x+sum(np.sum(g*d) for g,d in zip(dw,direction_w))+sum(g@d for g,d in zip(db,direction_b))
            fp=propagation(x+h*direction_x,[w+h*d for w,d in zip(W308,direction_w)],[b+h*d for b,d in zip(b308,direction_b)],kind)[0]
            fm=propagation(x-h*direction_x,[w-h*d for w,d in zip(W308,direction_w)],[b-h*d for b,d in zip(b308,direction_b)],kind)[0]
            assert abs((fp-fm)/(2*h)-predicted)<1e-7
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q309 复杂度与标量损失：不盲目构造完整 Jacobian

**新编，依据 Homework02 Exercise 2、Chapter3(1)向量输出问题、汇编 C3-05、09。**

实现 `network_cost(widths)` 返回 `(P,C)`，其中 $P=\sum(n_{l-1}n_l+n_l)$ 为参数量，
$C=\sum n_{l-1}n_l$ 为一次前向矩阵向量乘法的乘法数（不计偏置与激活的低阶开销）。
对 `3→2→4→6→8→10→1` 求P、C及全部输入/参数的中心差分前向调用次数。
再取线性向量输出 $y=Wx+b$、损失 $\ell=\sum_j(y_j-t_j)^2$，用 PyTorch 一次反传得到 `W` 梯度，
并用外积 $2(y-t)x^T$ 手算校准。回答输出宽度为m时，逐输出反传的完整Jacobian与标量损失反传有何不同。


### 自检接口／需保存的结果

- `network_cost(widths)` 按 `(P,C)` 顺序返回两个整数；对题面指定深网络的结果分别保存为 `P309,C309`，全部输入/参数中心差分的前向调用次数保存为整数 `fd_calls309`。
- 为统一向量输出实验，固定输入 `x309=[1,-2]`、权重 `W309=[[1,2],[-1,.5],[.2,.3]]`、偏置 `b309=[.1,-.2,.3]`、目标 `t309=[0,1,-1]`。四者均使用CPU float64张量；`W309,b309` 是启用梯度的叶张量，形状分别为 `(3,2),(3,)`，`x309,t309` 形状分别为 `(2,),(3,)`。
- 前向输出保存为张量 `y309`，形状 `(3,)`；完成题面标量损失的一次反传后，保留 `W309.grad,b309.grad`。手算得到的权重梯度保存为NumPy数组 `manual309`，形状 `(3,2)`，用来与自动微分比较。

这些是数据与输出接口约定，不要求使用参考答案的其他内部变量名。


<details><summary>提示（先尝试再展开）</summary>

复杂度首末项 `dM`、`d′M` 不能无条件省略。损失是标量时把误差归约也接进同一计算图。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    assert (P309,C309,fd_calls309)==(207,176,420)
    assert np.allclose(W309.grad.numpy(),manual309)
    assert np.allclose(b309.grad.numpy(),2*(y309.detach().numpy()-t309.numpy()))
    assert network_cost([1000,2,1000])==(5002,4000)
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")


## Q310 ReLU 分片线性逼近：实验不等于万能定理

**新编理解题，依据 Chapter3“万能近似定理”、9月16日第二三节、汇编 C3-10、11。**

在 `[0,1]` 上用 ReLU 网络实现 $x^2$ 的均匀网格分片线性插值。
对n等分令斜率 $m_j=(2j+1)/n$，网络可写成
$s_n(x)=m_0\operatorname{ReLU}(x)+\sum_{j=1}^{n-1}(m_j-m_{j-1})\operatorname{ReLU}(x-j/n)$。
实现 `square_relu(x,n)`；在1001点上比较n=4,8,16，打印最大绝对误差并保存 `output/Q310.png`。
说明本题给定构造的误差上界 $1/(4n^2)$，以及万能近似定理为何不保证当前优化器能找到参数。
图表使用英文标签。API以命令表为主，绘图关闭与目录创建是本地教学辅助。


### 自检接口／需保存的结果

`square_relu(x,n)` 返回与NumPy输入 `x` 同形状的数值数组。另保存字典 `errors310`：整数键为 `4,8,16`，每个值为对应n在题面1001点网格上的最大绝对误差（普通浮点数）。图片文件名保持题面要求；其他绘图变量可自定。


<details><summary>提示（先尝试再展开）</summary>

等距插值的相邻斜率增量恒为2/n。二次函数在每个子区间中点达到最大插值误差。


</details>


In [ ]:
# TODO：在此实现本题，再运行下一自检单元。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
try:
    for n in [4,8,16]:
        knots=np.linspace(0,1,n+1)
        assert np.max(np.abs(square_relu(knots,n)-knots**2))<1e-12
        assert errors310[n] <= 1/(4*n*n)+1e-12
    assert errors310[8]<errors310[4] and errors310[16]<errors310[8]
    assert Path('output/Q310.png').stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")
